In [1]:
# Load File
with open("the_verdict.txt","r",encoding="utf-8") as f:
    raw_text = f.read()[93:]

len(raw_text)

21842

In [2]:
print(raw_text[:99])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [3]:
import re
text = "Hello, world. This, is a test."
result = re.split(r'([,.:;?_!"()\']|--|\s)',text)
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [4]:
preprocessed =re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
preprocessed = [item for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [5]:
tokens = sorted(list(set(preprocessed)))
tokens.extend(["<|unk|>","<|endoftext|>"])
print(f"Size of tokens : {len(tokens)}")

Size of tokens : 1229


In [6]:
tokens_dict = {}

for token_id,token in enumerate(tokens):
    tokens_dict[token] = token_id

idtotoken = {v:k for k,v in tokens_dict.items()}

In [7]:
input_tokens = [tokens_dict[token] for token in preprocessed]
print(input_tokens[:30])
output_tokens = [idtotoken[id] for id in input_tokens]
print(output_tokens[:30])

[75, 66, 191, 1094, 80, 60, 904, 156, 303, 554, 7, 1093, 156, 568, 501, 456, 7, 996, 654, 1172, 786, 576, 1050, 1107, 736, 1107, 603, 1078, 6, 636]
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [8]:
class SimpleTokenizerV1:
    def __init__(self,vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in self.str_to_int.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',text)
        preprocessed = [item for item in preprocessed if item.strip()]
        output = [self.str_to_int[word] for word in preprocessed]
        return output
    
    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])

        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)   
        return text
    

tokenizer = SimpleTokenizerV1(vocab = tokens_dict)

text =  """"It's the last he painted, you know," 
       Mrs. Gisburn said with pardonable pride."""

ids = tokenizer.encode(text)
print(f"Ids : {ids}")

print(f"Decoded Ids : {tokenizer.decode(ids)}")

Ids : [1, 79, 2, 937, 1079, 671, 601, 826, 6, 1223, 665, 6, 1, 94, 8, 60, 938, 1204, 834, 875, 8]
Decoded Ids : " It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [9]:
class SimpleTokenizerV2:
    def __init__(self,vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in self.str_to_int.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',text)
        preprocessed = [item for item in preprocessed if item.strip()]
        preprocessed = [word if word in self.str_to_int else "<|unk|>" for word in preprocessed ]
        output = [self.str_to_int[word] for word in preprocessed]
        return output
    
    def decode(self,ids):
        text = " ".join([self.int_to_str[i] for i in ids])

        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)   
        return text

In [10]:
tokenizer = SimpleTokenizerV2(tokens_dict)
text = "Hello, do you like tea?"
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))

[1227, 6, 415, 1223, 700, 1064, 15]
<|unk|>, do you like tea?


In [11]:
from collections import defaultdict
def learn_bpe(corpus, num_merges=3):
   vocab = defaultdict(int)
   for word in corpus.split():
       chars = list(word)
       for i in range(len(chars) - 1):
           pair = (chars[i], chars[i + 1])
           vocab[pair] += 1
   merges = []
   for _ in range(num_merges):
       if not vocab:
           break
       most_frequent = max(vocab, key=vocab.get)
       merges.append(most_frequent)
       new_vocab = defaultdict(int)
       for pair, count in vocab.items():
           if pair == most_frequent:
               continue
           new_vocab[pair] = count
       vocab = new_vocab
   return merges
# Example usage
corpus = "ab bc bcd cde"
merges = learn_bpe(corpus, num_merges=5)
print("Learned Merges:", merges)

Learned Merges: [('b', 'c'), ('c', 'd'), ('a', 'b'), ('d', 'e')]


In [12]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))


5521


In [13]:
enc_sample = enc_text[50:]

In [28]:
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self,txt,tokenizer,maxlength,stride=1):
        super().__init__()
        token_ids= tokenizer.encode(txt)
        self.input_ids = []
        self.output_ids = []

        for i in range(0,len(token_ids)-maxlength,stride):
            self.input_ids.append(torch.tensor(token_ids[i:i+ maxlength]))
            self.output_ids.append(torch.tensor(token_ids[i+1:i+ maxlength+1]))

    
    def __getitem__(self, index):
        return self.input_ids[index], self.output_ids[index]
    
    def __len__(self):
        return len(self.output_ids)
    



def create_dataloader_v1(txt,maxlength=256,stride=128,batch_size=4,num_workers=0,shuffle=True,drop_last=True):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt,tokenizer,maxlength,stride)
    dl = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dl

dataloader = create_dataloader_v1(raw_text,maxlength=4,batch_size=1,stride=1,shuffle=False)
data_iter = iter(dataloader)
firstbatch = next(data_iter)
print(firstbatch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [35]:
secondbatch = next(data_iter)
print(secondbatch)

[tensor([[  402,   271, 10899,  2138]]), tensor([[  271, 10899,  2138,   257]])]


In [37]:
dataloader2 = create_dataloader_v1(raw_text,maxlength=4,stride=4,batch_size=8,shuffle=False)
data_iter = iter(dataloader2)
next(data_iter)

[tensor([[   40,   367,  2885,  1464],
         [ 1807,  3619,   402,   271],
         [10899,  2138,   257,  7026],
         [15632,   438,  2016,   257],
         [  922,  5891,  1576,   438],
         [  568,   340,   373,   645],
         [ 1049,  5975,   284,   502],
         [  284,  3285,   326,    11]]),
 tensor([[  367,  2885,  1464,  1807],
         [ 3619,   402,   271, 10899],
         [ 2138,   257,  7026, 15632],
         [  438,  2016,   257,   922],
         [ 5891,  1576,   438,   568],
         [  340,   373,   645,  1049],
         [ 5975,   284,   502,   284],
         [ 3285,   326,    11,   287]])]